# 3D-LLM Single-GPU Probing Benchmark

## Goal
This notebook evaluates the released **3D-LLM** checkpoint in a **single-GPU inference-only setting**.  
The objective is **not** to reproduce the original training pipeline, but to probe what the model actually relies on at inference time:
- semantic point features,
- feature-to-point alignment,
- and prompt phrasing.

## Experimental setup
We evaluate the model on a fixed subset of **$N=60$ Objaverse objects**.  
For each object $o$, the input consists of:
- a point cloud $P_o \in \mathbb{R}^{n \times 3}$,
- aligned point features $F_o \in \mathbb{R}^{n \times d}$,

with at most **$n=4096$ points** per object.  
Generations are produced with a fixed maximum decoding length **$L_{\max}=32$**.

## Probes

### 1. Feature robustness
The point coordinates are kept fixed, while the input features are modified with controlled perturbations:
- `baseline`
- `gaussian_005`
- `gaussian_020`
- `shuffle_feat`
- `zero_feat`

The goal is to test whether the model depends on:
- precise local feature geometry,
- or mainly on global semantic feature content.

### 2. Prompt robustness
The 3D input is kept fixed, while the text query is replaced by semantically similar paraphrases.  
We consider prompts from two groups:
- **object identity**
- **object description**

Within each group, the first phrasing is treated as the **baseline prompt**.

## Evaluation protocol
For each object $o$, prompt $q$, and perturbation $v$, the model produces an answer $y_{o,q,v}$.  
This answer is compared to the corresponding baseline answer $y_{o,q,0}$.

We report average agreement over the object set:
$
\mathrm{Score}(q,v)=\frac{1}{N}\sum_{o=1}^{N} m\!\left(y_{o,q,v},\,y_{o,q,0}\right),
$
where $m$ is one of the following metrics:
- **Exact Match (EM)**,
- **Sequence Similarity (Seq. Sim.)**,
- **Jaccard similarity**.

## Metrics
Let $y_b$ be the baseline answer and $y$ the answer under perturbation.

- **Exact Match**
$
\mathrm{EM}(y_b,y)=
\begin{cases}
1 & \text{if } y_b = y,\\
0 & \text{otherwise.}
\end{cases}
$

- **Sequence Similarity**
$
\mathrm{SeqSim}(y_b,y)\in[0,1],
$
computed from the normalized strings using the standard sequence-matching ratio.  
A value of $1$ means identical outputs.

- **Jaccard Similarity**
If $T(y)$ denotes the set of unique tokens in $y$, then
$
\mathrm{Jaccard}(y_b,y)=\frac{|T(y_b)\cap T(y)|}{|T(y_b)\cup T(y)|}.
$

## Interpretation
This benchmark is designed to answer a simple question:

> Does the released 3D-LLM checkpoint behave like a model that truly depends on 3D structure, or mostly like a model driven by semantic point features and prompt wording?

The results should therefore be interpreted as **probing evidence**, not as a full benchmark replication of the original paper.

In [25]:
# ==== Colab / environment sanity check ====
import os
import sys
import gc
import json
import random
import shutil
import zipfile
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 123
random.seed(SEED)
np.random.seed(SEED)

ROOT_PARENT = Path("/content")
REPO_DIR = ROOT_PARENT / "3D-LLM"
WORK_DIR = REPO_DIR / "3DLLM_BLIP2-base"
OUTPUT_DIR = WORK_DIR / "minimal_rigorous_results"

CKPT_URL = "https://drive.google.com/file/d/1tiis8mOdZGBzmR7vgZtRE4Ni_2FE4nTr/view?usp=drive_link"
OBJ_SUBSET_URL = "https://drive.google.com/file/d/1mJZONfWREfIUAPYXP65D65uS2EoplAfR/view?usp=drive_link"

CKPT_FILENAME = "pretrain_blip2_sam_flant5xl_v2.pth"
OBJ_ZIP_FILENAME = "objaverse_feat_subset.zip"

MAX_POINTS = 4096
MAX_LEN = 34
N_OBJECTS = 60

USE_FP16_MODEL = True
USE_COMPILE = False
TORCH_NUM_THREADS = 2

assert "google.colab" in sys.modules, "Please run this notebook inside Google Colab."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("WORK_DIR:", WORK_DIR)


WORK_DIR: /content/3D-LLM/3DLLM_BLIP2-base


In [2]:
# ==== Make sure a GPU is attached ====
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, go to Runtime > Change runtime type > GPU.")

DEVICE = "cuda"
print("GPU detected:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)


GPU detected: Tesla T4
CUDA version: 12.8


In [3]:
# ==== Fresh clone of the repo ====
%cd /content

if REPO_DIR.exists():
    print("Removing existing repo to start from a clean state...")
    shutil.rmtree(REPO_DIR)

!git clone --depth 1 https://github.com/gabriellenain/3D-LLM

assert WORK_DIR.exists(), f"Expected {WORK_DIR} to exist after cloning."
print("Repo cloned to:", WORK_DIR)


/content
Removing existing repo to start from a clean state...
Cloning into '3D-LLM'...
remote: Enumerating objects: 539, done.
remote: Counting objects: 100% (539/539), done.
remote: Compressing objects: 100% (326/326), done.
remote: Total 539 (delta 236), reused 362 (delta 191), pack-reused 0 (from 0)
Receiving objects: 100% (539/539), 35.51 MiB | 19.80 MiB/s, done.
Resolving deltas: 100% (236/236), done.
Repo cloned to: /content/3D-LLM/3DLLM_BLIP2-base


In [4]:
# ==== Install dependencies needed for inference ====
%cd /content/3D-LLM/3DLLM_BLIP2-base

!pip -q install --upgrade pip
!pip -q install gdown iopath fvcore positional-encodings omegaconf einops sentencepiece
!pip -q install --no-cache-dir "transformers==4.49.0"
!pip -q install pandas pyyaml requests scikit-learn matplotlib fairscale


/content/3D-LLM/3DLLM_BLIP2-base


In [5]:
# ==== Download checkpoint + Objaverse subset ====
%cd /content/3D-LLM/3DLLM_BLIP2-base

import gdown

ckpt_path = WORK_DIR / CKPT_FILENAME
obj_zip_path = WORK_DIR / OBJ_ZIP_FILENAME

if not ckpt_path.exists():
    print("Downloading checkpoint...")
    gdown.download(CKPT_URL, str(ckpt_path), quiet=False, fuzzy=True)
else:
    print("Checkpoint already exists:", ckpt_path)

if not obj_zip_path.exists():
    print("Downloading Objaverse subset zip...")
    gdown.download(OBJ_SUBSET_URL, str(obj_zip_path), quiet=False, fuzzy=True)
else:
    print("Objaverse subset zip already exists:", obj_zip_path)

assert ckpt_path.exists(), f"Checkpoint not found at {ckpt_path}"
assert obj_zip_path.exists(), f"Objaverse subset zip not found at {obj_zip_path}"


/content/3D-LLM/3DLLM_BLIP2-base


Downloading...
From (original): https://drive.google.com/uc?id=1tiis8mOdZGBzmR7vgZtRE4Ni_2FE4nTr
From (redirected): https://drive.google.com/uc?id=1tiis8mOdZGBzmR7vgZtRE4Ni_2FE4nTr&confirm=t&uuid=0741dee5-6d4e-4dee-aeeb-ed43247a65eb
To: /content/3D-LLM/3DLLM_BLIP2-base/pretrain_blip2_sam_flant5xl_v2.pth
100%|██████████| 4.47G/4.47G [01:06<00:00, 66.9MB/s]


Downloading...
From (original): https://drive.google.com/uc?id=1mJZONfWREfIUAPYXP65D65uS2EoplAfR
From (redirected): https://drive.google.com/uc?id=1mJZONfWREfIUAPYXP65D65uS2EoplAfR&confirm=t&uuid=e1afd260-c1b1-4c21-8f72-cce23bec35c3
To: /content/3D-LLM/3DLLM_BLIP2-base/objaverse_feat_subset.zip
100%|██████████| 1.83G/1.83G [00:19<00:00, 91.9MB/s]


In [6]:
# ==== Patch lavis/__init__.py and Qformer for modern transformers ====
lavis_init = WORK_DIR / "lavis" / "__init__.py"
lavis_init_backup = WORK_DIR / "lavis" / "__init___backup_colab.py"

if not lavis_init_backup.exists():
    shutil.copy(lavis_init, lavis_init_backup)

patched_lavis_init = """\"\"\"Minimal Colab patch for 3D-LLM inference.\"\"\"
import os
import sys
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models import *
from lavis.processors import *

root_dir = os.path.dirname(os.path.abspath(__file__))
default_cfg = OmegaConf.load(os.path.join(root_dir, "configs/default.yaml"))

registry.register_path("library_root", root_dir)
repo_root = os.path.join(root_dir, "..")
registry.register_path("repo_root", repo_root)

cache_root = os.path.join(repo_root, default_cfg.env.cache_root)
registry.register_path("cache_root", cache_root)

registry.register("MAX_INT", sys.maxsize)
registry.register("SPLIT_NAMES", ["train", "val", "test"])
"""
lavis_init.write_text(patched_lavis_init)

qformer_path = WORK_DIR / "lavis" / "models" / "blip2_models" / "Qformer.py"
txt = qformer_path.read_text()

old_block = """from transformers.modeling_utils import (
    PreTrainedModel,
    apply_chunking_to_forward,
    find_pruneable_heads_and_indices,
    prune_linear_layer,
)"""

new_block = """from transformers.modeling_utils import PreTrainedModel
from transformers.pytorch_utils import (
    apply_chunking_to_forward,
    find_pruneable_heads_and_indices,
    prune_linear_layer,
)"""

if new_block in txt:
    print("Qformer already patched")
elif old_block in txt:
    txt = txt.replace(old_block, new_block)
    qformer_path.write_text(txt)
    print("Qformer patched")
else:
    print("Expected import block not found; leaving file unchanged")

print("Patched lavis init and checked Qformer.")


Qformer patched
Patched lavis init and checked Qformer.


In [7]:
# ==== Extract Objaverse subset and normalize directory structure ====
extract_dir = WORK_DIR / "data"
extract_dir.mkdir(parents=True, exist_ok=True)

resolved_feat_root = None
preferred_root = extract_dir / "objaverse_feat"
if (preferred_root / "features").exists() and (preferred_root / "points").exists():
    resolved_feat_root = preferred_root

if resolved_feat_root is None:
    print("Extracting subset zip...")
    with zipfile.ZipFile(WORK_DIR / OBJ_ZIP_FILENAME, "r") as zf:
        zf.extractall(extract_dir)

    candidates = [p for p in extract_dir.rglob("*") if p.is_dir() and (p / "features").exists() and (p / "points").exists()]
    if not candidates:
        raise RuntimeError("Could not find an extracted directory containing both 'features' and 'points'.")
    candidates = sorted(candidates, key=lambda p: (("objaverse" not in p.name.lower()), len(str(p))))
    resolved_feat_root = candidates[0]

target_feat_root = extract_dir / "objaverse_feat"
if resolved_feat_root != target_feat_root:
    if target_feat_root.exists():
        shutil.rmtree(target_feat_root)
    shutil.move(str(resolved_feat_root), str(target_feat_root))

assert (target_feat_root / "features").exists()
assert (target_feat_root / "points").exists()

v1_alias = WORK_DIR / "pretrain_blip2_sam_flant5xl_v1.pth"
v2_ckpt = WORK_DIR / "pretrain_blip2_sam_flant5xl_v2.pth"
if not v1_alias.exists():
    try:
        v1_alias.symlink_to(v2_ckpt.name)
    except Exception:
        shutil.copy(v2_ckpt, v1_alias)

print("Resolved feature root:", target_feat_root)
print("v2 checkpoint path:", v2_ckpt)


Extracting subset zip...
Resolved feature root: /content/3D-LLM/3DLLM_BLIP2-base/data/objaverse_feat
v2 checkpoint path: /content/3D-LLM/3DLLM_BLIP2-base/pretrain_blip2_sam_flant5xl_v2.pth


In [8]:
# ==== Import lavis cleanly ====
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

for k in list(sys.modules.keys()):
    if k.startswith("lavis"):
        del sys.modules[k]

importlib.invalidate_caches()

import transformers
print("transformers version:", transformers.__version__)

import lavis
print("lavis imported successfully")


transformers version: 4.49.0


/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


lavis imported successfully


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [9]:
# ==== Lean model loading utilities ====
from omegaconf import OmegaConf
from lavis.common.registry import registry
import torch.nn as nn
torch.set_num_threads(TORCH_NUM_THREADS)
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def smart_resize_t5_embeddings_if_needed(model, state_dict):
    target_vocab = None
    for key in [
        "t5_model.shared.weight",
        "t5_model.encoder.embed_tokens.weight",
        "t5_model.decoder.embed_tokens.weight",
        "t5_model.lm_head.weight",
    ]:
        if key in state_dict:
            target_vocab = state_dict[key].shape[0]
            break

    if target_vocab is None:
        return

    current_vocab = model.t5_model.shared.weight.shape[0]
    if current_vocab == target_vocab:
        return

    tokenizer_len = len(model.t5_tokenizer)
    if tokenizer_len < target_vocab:
        extra_needed = target_vocab - tokenizer_len
        dummy_tokens = [f"<colab_fix_tok_{i}>" for i in range(extra_needed)]
        model.t5_tokenizer.add_special_tokens({"additional_special_tokens": dummy_tokens})

    model.t5_model.resize_token_embeddings(target_vocab)

def load_3dllm_model(ckpt_path=None, device=DEVICE):
    clean_memory()

    ckpt_path = Path(ckpt_path or (WORK_DIR / CKPT_FILENAME))
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

    model_cfg = OmegaConf.create({
        "arch": "blip2_t5",
        "model_type": "pretrain_flant5xl",
        "use_grad_checkpoint": False,
    })
    processor_cfg = OmegaConf.create({
        "name": "blip_question",
        "prompt": "",
    })

    print("Instantiating model...")
    model_cls = registry.get_model_class(model_cfg.arch)
    model = model_cls.from_pretrained(model_type=model_cfg.model_type)

    print("Loading checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint

    smart_resize_t5_embeddings_if_needed(model, state_dict)
    msg = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", len(msg.missing_keys), "| Unexpected keys:", len(msg.unexpected_keys))

    del checkpoint, state_dict
    clean_memory()

    model.eval()

    # enlever la tour vision inutile
    if hasattr(model, "visual_encoder"):
        del model.visual_encoder
    if hasattr(model, "ln_vision"):
        del model.ln_vision
    clean_memory()

    # tout passer en float16 AVANT de l'envoyer sur le GPU
    model.t5_model = model.t5_model.half().to(device)
    model.Qformer = model.Qformer.half().to(device)
    model.t5_proj = model.t5_proj.half().to(device)
    model.query_tokens = nn.Parameter(
        model.query_tokens.half().to(device),
        requires_grad=False
    )

    if isinstance(model.pos_embedding, torch.Tensor):
        model.pos_embedding = model.pos_embedding.half().to(device)

    # pos_embedding in this repo is already created with .cuda(), but make it explicit/safe
    if isinstance(model.pos_embedding, torch.Tensor):
        model.pos_embedding = model.pos_embedding.to(device)

    text_processor = registry.get_processor_class(processor_cfg.name).from_config(processor_cfg)
    clean_memory()

    if torch.cuda.is_available():
        print("Allocated GPU memory (GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))
        print("Reserved GPU memory (GB):", round(torch.cuda.memory_reserved() / 1024**3, 2))

    return model, text_processor

model, text_processor = load_3dllm_model()
print("Model loaded on", DEVICE)


Instantiating model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint...
Missing keys: 1067 | Unexpected keys: 0
Allocated GPU memory (GB): 5.78
Reserved GPU memory (GB): 6.24
Model loaded on cuda


In [26]:
# ==== Data loading + official-style inference helpers ====
import re
from difflib import SequenceMatcher
import matplotlib.pyplot as plt

OBJ_ID_PATH = WORK_DIR / "assets" / "objaverse_subset_ids_100.json"
OBJ_FEAT_ROOT = WORK_DIR / "data" / "objaverse_feat"

assert OBJ_ID_PATH.exists(), f"Missing {OBJ_ID_PATH}"
assert (OBJ_FEAT_ROOT / "features").exists(), "Missing feature directory"
assert (OBJ_FEAT_ROOT / "points").exists(), "Missing point directory"

def text_norm(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s

def exact_match(a: str, b: str) -> int:
    return int(text_norm(a) == text_norm(b))

def seq_similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, text_norm(a), text_norm(b)).ratio()

def jaccard_words(a: str, b: str) -> float:
    sa = set(text_norm(a).split())
    sb = set(text_norm(b).split())
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)

def subsample_pair(pc_feat, pc_points, max_points=None, fraction=1.0, seed=SEED):
    assert pc_feat.shape[0] == pc_points.shape[0]
    n = pc_feat.shape[0]

    if fraction <= 0 or fraction > 1:
        raise ValueError("fraction must be in (0, 1].")

    target = max(1, int(round(n * fraction)))
    if max_points is not None:
        target = min(target, max_points)

    if target >= n:
        return pc_feat.contiguous(), pc_points.contiguous()

    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(n, size=target, replace=False))
    idx_t = torch.from_numpy(idx).long()
    return pc_feat[idx_t].contiguous(), pc_points[idx_t].contiguous()

def get_all_valid_object_ids():
    with open(OBJ_ID_PATH, "r") as f:
        obj_ids = json.load(f)
    valid = []
    for obj_id in obj_ids:
        feat_path = OBJ_FEAT_ROOT / "features" / f"{obj_id}_outside.pt"
        pts_path = OBJ_FEAT_ROOT / "points" / f"{obj_id}_outside.npy"
        if feat_path.exists() and pts_path.exists():
            valid.append(obj_id)
    return valid

def choose_object_ids(n_objects=N_OBJECTS, seed=SEED):
    valid = get_all_valid_object_ids()
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(valid))[:min(n_objects, len(valid))]
    return [valid[i] for i in idx]

def load_object_sample(obj_id, max_points=MAX_POINTS, fraction=1.0, seed=SEED):
    feat_path = OBJ_FEAT_ROOT / "features" / f"{obj_id}_outside.pt"
    pts_path = OBJ_FEAT_ROOT / "points" / f"{obj_id}_outside.npy"

    if not feat_path.exists() or not pts_path.exists():
        raise FileNotFoundError(f"Missing files for {obj_id}")

    pc_feat = torch.load(feat_path, map_location="cpu", weights_only=False)
    if isinstance(pc_feat, np.ndarray):
        pc_feat = torch.from_numpy(pc_feat)
    pc_feat = pc_feat.to(dtype=torch.float16)

    pc_points = np.load(pts_path, mmap_mode="r")
    pc_points = torch.from_numpy(np.array(pc_points, copy=True)).to(dtype=torch.int32)

    pc_feat, pc_points = subsample_pair(pc_feat, pc_points, max_points=max_points, fraction=fraction, seed=seed)

    if not torch.isfinite(pc_feat.float()).all():
        raise ValueError(f"Non-finite feature values for {obj_id}")
    if not torch.isfinite(pc_points.float()).all():
        raise ValueError(f"Non-finite point values for {obj_id}")

    return obj_id, pc_feat.contiguous(), pc_points.contiguous()

@torch.inference_mode()
def answer_question_official(question, pc_feat, pc_points, max_len=MAX_LEN):
    prompt = text_processor(question)

    pc_feat_b = pc_feat.unsqueeze(0).to(device=DEVICE, dtype=torch.float16, non_blocking=True)
    pc_points_b = pc_points.unsqueeze(0).to(device=DEVICE, dtype=torch.long, non_blocking=True)

    samples = {
        "text_input": [prompt],
        "pc_feat": pc_feat_b,
        "pc": pc_points_b,
    }

    out = model.predict_answers(
        samples=samples,
        num_beams=1,
        max_len=max_len,
        min_len=1,
        repetition_penalty=1.0,
        length_penalty=1.0,
    )

    if isinstance(out, (list, tuple)):
        return str(out[0])
    return str(out)

def safe_answer(prompt, feat, pts, max_len=MAX_LEN):
    try:
        ans = answer_question_official(prompt, feat, pts, max_len=max_len)
        return {
            "status": "ok",
            "answer": ans,
            "error_msg": "",
        }
    except Exception as e:
        return {
            "status": "generation_error",
            "answer": "",
            "error_msg": f"{type(e).__name__}: {e}",
        }

object_ids = choose_object_ids(n_objects=N_OBJECTS, seed=SEED)
print("Selected", len(object_ids), "objects")
print(object_ids[:5])


Selected 60 objects
['48d3782909204430b57a36e1e3f922f2', '64998ee900d641d2b5096caaa5cdf006', 'b77bc3f4fb9d4bbd963db88fa0cc0aa3', '62fc5fec1aee4296b128f78cb296c6cd', '39bbe30b9af54eb584744a3ef97f1962']


In [ ]:
# ==== Cleaner paper version ====

import os
import numpy as np
import matplotlib.pyplot as plt

def normalize_points(pts):
    pts = np.asarray(pts).copy()
    pts = pts - pts.mean(axis=0, keepdims=True)
    scale = np.max(np.linalg.norm(pts, axis=1))
    if scale > 0:
        pts = pts / scale
    return pts

def save_clean_multiview_panel(pts, obj_id="object", save_path="multiview_clean.png", s=1.0):
    pts = normalize_points(pts)

    views = [
        (20, 30),
        (20, 120),
        (20, 210),
        (90, -90),
    ]

    fig = plt.figure(figsize=(10, 2.8))
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]

    for i, (elev, azim) in enumerate(views, 1):
        ax = fig.add_subplot(1, 4, i, projection="3d")
        ax.scatter(x, y, z, s=s)
        ax.view_init(elev=elev, azim=azim)

        lim = 1.05
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_zlim(-lim, lim)

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_zticks([])
        ax.set_box_aspect((1, 1, 1))
        ax.set_axis_off()

    plt.tight_layout(pad=0.05)
    plt.savefig(save_path, dpi=400, bbox_inches="tight", pad_inches=0.02)
    plt.show()


obj_id = object_ids[0]
_, feat, pts = load_object_sample(obj_id, max_points=MAX_POINTS, fraction=1.0, seed=SEED)
pts_np = pts.detach().cpu().numpy() if hasattr(pts, "detach") else np.asarray(pts)

save_clean_multiview_panel(
    pts_np,
    obj_id=obj_id,
    save_path=f"multiview_clean_{obj_id}.png",
    s=1.0,
)

In [27]:
# ==== Quick sanity check on one object ====
SANITY_PROMPTS = [
    "What is this object?",
    "Give a concise description of this 3D object.",
]

obj_id, feat, pts = load_object_sample(object_ids[0], max_points=MAX_POINTS, fraction=1.0, seed=SEED)
print("Object:", obj_id)
print("feat shape:", tuple(feat.shape), "dtype:", feat.dtype)
print("pts shape :", tuple(pts.shape),  "dtype:", pts.dtype)

for prompt in SANITY_PROMPTS:
    out = safe_answer(prompt, feat, pts, max_len=MAX_LEN)
    print("\nPROMPT:", prompt)
    print("STATUS:", out["status"])
    print("ANSWER:", out["answer"] if out["answer"] else out["error_msg"])


Object: 48d3782909204430b57a36e1e3f922f2
feat shape: (4096, 1408) dtype: torch.float16
pts shape : (4096, 3) dtype: torch.int32

PROMPT: What is this object?
STATUS: ok
ANSWER: <pad> A white square.</s>

PROMPT: Give a concise description of this 3D object.
STATUS: ok
ANSWER: <pad> A white square is a rectangular prism that is a solid object that is not a solid object. It is a solid object that is not a


In [28]:
# ==== Feature corruptions only (pc is kept untouched) ====
def corrupt_features(pc_feat: torch.Tensor, variant: str, seed: int = SEED) -> torch.Tensor:
    g = torch.Generator(device="cpu")
    g.manual_seed(seed)
    feat = pc_feat.clone()

    if variant == "baseline":
        return feat

    if variant == "shuffle_feat":
        perm = torch.randperm(feat.shape[0], generator=g)
        return feat[perm].contiguous()

    if variant == "gaussian_005":
        sigma = 0.05 * feat.float().std().clamp_min(1e-6)
        noise = torch.randn(feat.shape, generator=g, dtype=torch.float32) * sigma
        return (feat.float() + noise).to(dtype=feat.dtype).contiguous()

    if variant == "gaussian_020":
        sigma = 0.20 * feat.float().std().clamp_min(1e-6)
        noise = torch.randn(feat.shape, generator=g, dtype=torch.float32) * sigma
        return (feat.float() + noise).to(dtype=feat.dtype).contiguous()

    if variant == "zero_feat":
        return torch.zeros_like(feat).contiguous()

    raise ValueError(f"Unknown corruption variant: {variant}")

FEATURE_VARIANTS = ["baseline", "gaussian_005", "gaussian_020", "shuffle_feat", "zero_feat"]
ROBUSTNESS_PROMPTS = [
    "What is this object?",
    "Give a concise description of this 3D object.",
]

def run_feature_robustness(object_ids, prompts, variants, max_points=MAX_POINTS):
    rows = []

    for obj_id in object_ids:
        print(f"[object] {obj_id}")
        try:
            _, feat_orig, pts_orig = load_object_sample(obj_id, max_points=max_points, fraction=1.0, seed=SEED)
        except Exception as e:
            rows.append({
                "obj_id": obj_id,
                "prompt": "[load_failure]",
                "variant": "[load_failure]",
                "status": "load_error",
                "error_msg": f"{type(e).__name__}: {e}",
            })
            continue

        baseline_cache = {}
        for prompt in prompts:
            baseline_cache[prompt] = safe_answer(prompt, feat_orig, pts_orig, max_len=MAX_LEN)

        for variant in variants:
            feat_var = corrupt_features(feat_orig, variant, seed=SEED)

            for prompt in prompts:
                out = safe_answer(prompt, feat_var, pts_orig, max_len=MAX_LEN)
                base = baseline_cache[prompt]
                both_ok = int(base["status"] == "ok" and out["status"] == "ok")

                rows.append({
                    "obj_id": obj_id,
                    "prompt": prompt,
                    "variant": variant,
                    "baseline_answer": base["answer"],
                    "baseline_status": base["status"],
                    "baseline_error_msg": base["error_msg"],
                    "answer": out["answer"],
                    "status": out["status"],
                    "error_msg": out["error_msg"],
                    "both_ok": both_ok,
                    "exact_match_to_baseline": exact_match(base["answer"], out["answer"]) if both_ok else np.nan,
                    "seq_similarity_to_baseline": seq_similarity(base["answer"], out["answer"]) if both_ok else np.nan,
                    "jaccard_to_baseline": jaccard_words(base["answer"], out["answer"]) if both_ok else np.nan,
                    "answer_len": len(out["answer"].split()) if out["answer"] else 0,
                })

    return pd.DataFrame(rows)

df_feat = run_feature_robustness(object_ids, ROBUSTNESS_PROMPTS, FEATURE_VARIANTS, max_points=MAX_POINTS)
print("Rows:", len(df_feat))
display(df_feat.head(10))


[object] 48d3782909204430b57a36e1e3f922f2
[object] 64998ee900d641d2b5096caaa5cdf006
[object] b77bc3f4fb9d4bbd963db88fa0cc0aa3
[object] 62fc5fec1aee4296b128f78cb296c6cd
[object] 39bbe30b9af54eb584744a3ef97f1962
[object] 64297c2df9094a7bbee3b1fa09058427
[object] 6a65ab62e6cd4a8fa6df61eca9af84e3
[object] b94170f1f2fb4a95a1cb779fa4391c43
[object] 1149e99c216a41d6a8d41a0ee99b6902
[object] 4135d9e904684df1b467b15bd0b289a3
[object] e036fd6204954595839ab65635ba641d
[object] 25b5d2e833e748249f0ed335fc196acd
[object] b8e9454ad9f54317937fbb3ad10c2051
[object] 40ae118e286246e9a813a3a1fc97c8ac
[object] 2bbdade09abc4ebc85cb49194364e4a2
[object] a5789477743a45f7a0e046dc0bd25268
[object] 3cdb6b2dcc274e05b8dee1497ec6a54b
[object] 24bc60727c5846d3b17543efd23711f2
[object] 4553469a17f642008b6b7d3647a1d2e6
[object] b5d7683964734fc08a7090e9b3473a90
[object] 662732cd375648bca0323a3e4d982012
[object] e53ec4ab884d4264a54cbbacc8b6df9e
[object] 2fdfe3d9551d4ed088fc1173498456ed
[object] 2c7934a650eb46f08ad3902f8

,obj_id,prompt,variant,baseline_answer,baseline_status,baseline_error_msg,answer,status,error_msg,both_ok,exact_match_to_baseline,seq_similarity_to_baseline,jaccard_to_baseline,answer_len
0,48d3782909204430b57a36e1e3f922f2,What is this object?,baseline,<pad> A white square.</s>,ok,,<pad> A white square.</s>,ok,,1,1,1.000000,1.000000,4
1,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,baseline,<pad> A white square is a rectangular prism th...,ok,,<pad> A white square is a rectangular prism th...,ok,,1,1,1.000000,1.000000,28
2,48d3782909204430b57a36e1e3f922f2,What is this object?,gaussian_005,<pad> A white square.</s>,ok,,<pad> A white square.</s>,ok,,1,1,1.000000,1.000000,4
3,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,gaussian_005,<pad> A white square is a rectangular prism th...,ok,,<pad> A white square is a rectangular prism th...,ok,,1,1,1.000000,1.000000,28
4,48d3782909204430b57a36e1e3f922f2,What is this object?,gaussian_020,<pad> A white square.</s>,ok,,<pad> A white square.</s>,ok,,1,1,1.000000,1.000000,4
5,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,gaussian_020,<pad> A white square is a rectangular prism th...,ok,,<pad> A white square is a rectangular prism th...,ok,,1,1,1.000000,1.000000,28
6,48d3782909204430b57a36e1e3f922f2,What is this object?,shuffle_feat,<pad> A white square.</s>,ok,,<pad> A white square.</s>,ok,,1,1,1.000000,1.000000,4
7,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,shuffle_feat,<pad> A white square is a rectangular prism th...,ok,,<pad> A white square is a rectangular prism th...,ok,,1,1,1.000000,1.000000,28
8,48d3782909204430b57a36e1e3f922f2,What is this object?,zero_feat,<pad> A white square.</s>,ok,,<pad> a teddy bear</s>,ok,,1,0,0.666667,0.333333,4
9,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,zero_feat,<pad> A white square is a rectangular prism th...,ok,,<pad> a 3D object that is not a real object</s>,ok,,1,0,0.452381,0.400000,10


In [13]:
# ==== Summary: feature robustness ====
def summarize_feature_robustness(df):
    tmp = df.copy()
    tmp = tmp[tmp["variant"] != "[load_failure]"].copy()

    return (
        tmp.groupby(["prompt", "variant"], dropna=False)
           .agg(
               n=("obj_id", "count"),
               generation_ok_rate=("status", lambda s: np.mean(np.array(s) == "ok")),
               both_ok_rate=("both_ok", "mean"),
               exact_match_mean=("exact_match_to_baseline", "mean"),
               seq_similarity_mean=("seq_similarity_to_baseline", "mean"),
               jaccard_mean=("jaccard_to_baseline", "mean"),
               answer_len_mean=("answer_len", "mean"),
           )
           .reset_index()
           .sort_values(["prompt", "variant"])
    )

summary_feat = summarize_feature_robustness(df_feat)
display(summary_feat)


,prompt,variant,n,generation_ok_rate,both_ok_rate,exact_match_mean,seq_similarity_mean,jaccard_mean,answer_len_mean
0,Give a concise description of this 3D object.,baseline,8,1.0,1.0,1.000,1.000000,1.000000,13.625
1,Give a concise description of this 3D object.,gaussian_005,8,1.0,1.0,1.000,1.000000,1.000000,13.625
2,Give a concise description of this 3D object.,gaussian_020,8,1.0,1.0,1.000,1.000000,1.000000,13.625
3,Give a concise description of this 3D object.,shuffle_feat,8,1.0,1.0,0.875,0.974099,0.951923,13.750
4,Give a concise description of this 3D object.,zero_feat,8,1.0,1.0,0.000,0.402993,0.186467,10.000
5,What is this object?,baseline,8,1.0,1.0,1.000,1.000000,1.000000,11.000
6,What is this object?,gaussian_005,8,1.0,1.0,1.000,1.000000,1.000000,11.000
7,What is this object?,gaussian_020,8,1.0,1.0,0.750,0.953470,0.894643,10.625
8,What is this object?,shuffle_feat,8,1.0,1.0,0.875,0.995902,0.975000,11.000
9,What is this object?,zero_feat,8,1.0,1.0,0.000,0.345276,0.201894,4.000


In [14]:
# ==== Prompt robustness benchmark ====
PARAPHRASE_GROUPS = {
    "object_identity": [
        "What is this object?",
        "What does this 3D object represent?",
        "Name the object in one short phrase.",
    ],
    "object_description": [
        "Give a concise description of this 3D object.",
        "Describe this 3D object briefly.",
        "What does this object look like?",
    ],
}

def run_prompt_robustness(object_ids, paraphrase_groups, max_points=MAX_POINTS):
    rows = []

    for obj_id in object_ids:
        print(f"[prompt robustness] {obj_id}")
        try:
            _, feat, pts = load_object_sample(obj_id, max_points=max_points, fraction=1.0, seed=SEED)
        except Exception as e:
            rows.append({
                "obj_id": obj_id,
                "group": "[load_failure]",
                "prompt": "[load_failure]",
                "status": "load_error",
                "error_msg": f"{type(e).__name__}: {e}",
            })
            continue

        for group_name, prompts in paraphrase_groups.items():
            baseline_prompt = prompts[0]
            base = safe_answer(baseline_prompt, feat, pts, max_len=MAX_LEN)

            for prompt in prompts:
                out = safe_answer(prompt, feat, pts, max_len=MAX_LEN)
                both_ok = int(base["status"] == "ok" and out["status"] == "ok")

                rows.append({
                    "obj_id": obj_id,
                    "group": group_name,
                    "baseline_prompt": baseline_prompt,
                    "prompt": prompt,
                    "baseline_answer": base["answer"],
                    "baseline_status": base["status"],
                    "answer": out["answer"],
                    "status": out["status"],
                    "error_msg": out["error_msg"],
                    "both_ok": both_ok,
                    "exact_match_to_baseline": exact_match(base["answer"], out["answer"]) if both_ok else np.nan,
                    "seq_similarity_to_baseline": seq_similarity(base["answer"], out["answer"]) if both_ok else np.nan,
                    "jaccard_to_baseline": jaccard_words(base["answer"], out["answer"]) if both_ok else np.nan,
                })

    return pd.DataFrame(rows)

df_prompt = run_prompt_robustness(object_ids, PARAPHRASE_GROUPS, max_points=MAX_POINTS)
print("Rows:", len(df_prompt))
display(df_prompt.head(10))


[prompt robustness] 48d3782909204430b57a36e1e3f922f2
[prompt robustness] 64998ee900d641d2b5096caaa5cdf006
[prompt robustness] b77bc3f4fb9d4bbd963db88fa0cc0aa3
[prompt robustness] 62fc5fec1aee4296b128f78cb296c6cd
[prompt robustness] 39bbe30b9af54eb584744a3ef97f1962
[prompt robustness] 64297c2df9094a7bbee3b1fa09058427
[prompt robustness] 6a65ab62e6cd4a8fa6df61eca9af84e3
[prompt robustness] b94170f1f2fb4a95a1cb779fa4391c43
Rows: 48


,obj_id,group,baseline_prompt,prompt,baseline_answer,baseline_status,answer,status,error_msg,both_ok,exact_match_to_baseline,seq_similarity_to_baseline,jaccard_to_baseline
0,48d3782909204430b57a36e1e3f922f2,object_identity,What is this object?,What is this object?,<pad> A white square.</s>,ok,<pad> A white square.</s>,ok,,1,1,1.000000,1.000000
1,48d3782909204430b57a36e1e3f922f2,object_identity,What is this object?,What does this 3D object represent?,<pad> A white square.</s>,ok,<pad> It represents a white square.</s>,ok,,1,0,0.730769,0.666667
2,48d3782909204430b57a36e1e3f922f2,object_identity,What is this object?,Name the object in one short phrase.,<pad> A white square.</s>,ok,<pad> A white square.</s>,ok,,1,1,1.000000,1.000000
3,48d3782909204430b57a36e1e3f922f2,object_description,Give a concise description of this 3D object.,Give a concise description of this 3D object.,<pad> A white square is a rectangular prism th...,ok,<pad> A white square is a rectangular prism th...,ok,,1,1,1.000000,1.000000
4,48d3782909204430b57a36e1e3f922f2,object_description,Give a concise description of this 3D object.,Describe this 3D object briefly.,<pad> A white square is a rectangular prism th...,ok,<pad> A white square with a smooth surface and...,ok,,1,0,0.500000,0.210526
5,48d3782909204430b57a36e1e3f922f2,object_description,Give a concise description of this 3D object.,What does this object look like?,<pad> A white square is a rectangular prism th...,ok,<pad> It is a white square.</s>,ok,,1,0,0.319328,0.384615
6,64998ee900d641d2b5096caaa5cdf006,object_identity,What is this object?,What is this object?,<pad> A white moose with a horn and a snout.</s>,ok,<pad> A white moose with a horn and a snout.</s>,ok,,1,1,1.000000,1.000000
7,64998ee900d641d2b5096caaa5cdf006,object_identity,What is this object?,What does this 3D object represent?,<pad> A white moose with a horn and a snout.</s>,ok,<pad> This 3D object represents a 3D object.</s>,ok,,1,0,0.452381,0.153846
8,64998ee900d641d2b5096caaa5cdf006,object_identity,What is this object?,Name the object in one short phrase.,<pad> A white moose with a horn and a snout.</s>,ok,<pad> A white teddy bear with a hat and a hat....,ok,,1,0,0.720930,0.416667
9,64998ee900d641d2b5096caaa5cdf006,object_description,Give a concise description of this 3D object.,Give a concise description of this 3D object.,"<pad> A 3D model of a moose with a snout, a snout",ok,"<pad> A 3D model of a moose with a snout, a snout",ok,,1,1,1.000000,1.000000


In [15]:
# ==== Summary: prompt robustness ====
def summarize_prompt_robustness(df):
    tmp = df.copy()
    tmp = tmp[tmp["group"] != "[load_failure]"].copy()

    return (
        tmp.groupby(["group", "prompt"], dropna=False)
           .agg(
               n=("obj_id", "count"),
               generation_ok_rate=("status", lambda s: np.mean(np.array(s) == "ok")),
               both_ok_rate=("both_ok", "mean"),
               exact_match_mean=("exact_match_to_baseline", "mean"),
               seq_similarity_mean=("seq_similarity_to_baseline", "mean"),
               jaccard_mean=("jaccard_to_baseline", "mean"),
           )
           .reset_index()
           .sort_values(["group", "prompt"])
    )

summary_prompt = summarize_prompt_robustness(df_prompt)
display(summary_prompt)


,group,prompt,n,generation_ok_rate,both_ok_rate,exact_match_mean,seq_similarity_mean,jaccard_mean
0,object_description,Describe this 3D object briefly.,8,1.0,1.0,0.375,0.625833,0.501129
1,object_description,Give a concise description of this 3D object.,8,1.0,1.0,1.000,1.000000,1.000000
2,object_description,What does this object look like?,8,1.0,1.0,0.000,0.566421,0.372235
3,object_identity,Name the object in one short phrase.,8,1.0,1.0,0.125,0.644859,0.464286
4,object_identity,What does this 3D object represent?,8,1.0,1.0,0.000,0.486597,0.300069
5,object_identity,What is this object?,8,1.0,1.0,1.000,1.000000,1.000000


In [16]:
# ==== Diagnostics and save CSVs ====
def show_examples(df, n=20):
    cols = [c for c in [
        "obj_id", "group", "prompt", "variant",
        "baseline_answer", "answer", "status", "error_msg",
        "exact_match_to_baseline", "seq_similarity_to_baseline", "jaccard_to_baseline"
    ] if c in df.columns]
    return df[cols].head(n)

print("Feature robustness examples:")
display(show_examples(df_feat, n=20))

print("\nPrompt robustness examples:")
display(show_examples(df_prompt, n=20))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_feat.to_csv(OUTPUT_DIR / "feature_robustness_rows.csv", index=False)
summary_feat.to_csv(OUTPUT_DIR / "feature_robustness_summary.csv", index=False)

df_prompt.to_csv(OUTPUT_DIR / "prompt_robustness_rows.csv", index=False)
summary_prompt.to_csv(OUTPUT_DIR / "prompt_robustness_summary.csv", index=False)

print("\nSaved files:")
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p)


Feature robustness examples:


,obj_id,prompt,variant,baseline_answer,answer,status,error_msg,exact_match_to_baseline,seq_similarity_to_baseline,jaccard_to_baseline
0,48d3782909204430b57a36e1e3f922f2,What is this object?,baseline,<pad> A white square.</s>,<pad> A white square.</s>,ok,,1,1.000000,1.000000
1,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,baseline,<pad> A white square is a rectangular prism th...,<pad> A white square is a rectangular prism th...,ok,,1,1.000000,1.000000
2,48d3782909204430b57a36e1e3f922f2,What is this object?,gaussian_005,<pad> A white square.</s>,<pad> A white square.</s>,ok,,1,1.000000,1.000000
3,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,gaussian_005,<pad> A white square is a rectangular prism th...,<pad> A white square is a rectangular prism th...,ok,,1,1.000000,1.000000
4,48d3782909204430b57a36e1e3f922f2,What is this object?,gaussian_020,<pad> A white square.</s>,<pad> A white square.</s>,ok,,1,1.000000,1.000000
5,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,gaussian_020,<pad> A white square is a rectangular prism th...,<pad> A white square is a rectangular prism th...,ok,,1,1.000000,1.000000
6,48d3782909204430b57a36e1e3f922f2,What is this object?,shuffle_feat,<pad> A white square.</s>,<pad> A white square.</s>,ok,,1,1.000000,1.000000
7,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,shuffle_feat,<pad> A white square is a rectangular prism th...,<pad> A white square is a rectangular prism th...,ok,,1,1.000000,1.000000
8,48d3782909204430b57a36e1e3f922f2,What is this object?,zero_feat,<pad> A white square.</s>,<pad> a teddy bear</s>,ok,,0,0.666667,0.333333
9,48d3782909204430b57a36e1e3f922f2,Give a concise description of this 3D object.,zero_feat,<pad> A white square is a rectangular prism th...,<pad> a 3D object that is not a real object</s>,ok,,0,0.544118,0.400000



Prompt robustness examples:


,obj_id,group,prompt,baseline_answer,answer,status,error_msg,exact_match_to_baseline,seq_similarity_to_baseline,jaccard_to_baseline
0,48d3782909204430b57a36e1e3f922f2,object_identity,What is this object?,<pad> A white square.</s>,<pad> A white square.</s>,ok,,1,1.000000,1.000000
1,48d3782909204430b57a36e1e3f922f2,object_identity,What does this 3D object represent?,<pad> A white square.</s>,<pad> It represents a white square.</s>,ok,,0,0.730769,0.666667
2,48d3782909204430b57a36e1e3f922f2,object_identity,Name the object in one short phrase.,<pad> A white square.</s>,<pad> A white square.</s>,ok,,1,1.000000,1.000000
3,48d3782909204430b57a36e1e3f922f2,object_description,Give a concise description of this 3D object.,<pad> A white square is a rectangular prism th...,<pad> A white square is a rectangular prism th...,ok,,1,1.000000,1.000000
4,48d3782909204430b57a36e1e3f922f2,object_description,Describe this 3D object briefly.,<pad> A white square is a rectangular prism th...,<pad> A white square with a smooth surface and...,ok,,0,0.500000,0.210526
5,48d3782909204430b57a36e1e3f922f2,object_description,What does this object look like?,<pad> A white square is a rectangular prism th...,<pad> It is a white square.</s>,ok,,0,0.319328,0.384615
6,64998ee900d641d2b5096caaa5cdf006,object_identity,What is this object?,<pad> A white moose with a horn and a snout.</s>,<pad> A white moose with a horn and a snout.</s>,ok,,1,1.000000,1.000000
7,64998ee900d641d2b5096caaa5cdf006,object_identity,What does this 3D object represent?,<pad> A white moose with a horn and a snout.</s>,<pad> This 3D object represents a 3D object.</s>,ok,,0,0.452381,0.153846
8,64998ee900d641d2b5096caaa5cdf006,object_identity,Name the object in one short phrase.,<pad> A white moose with a horn and a snout.</s>,<pad> A white teddy bear with a hat and a hat....,ok,,0,0.720930,0.416667
9,64998ee900d641d2b5096caaa5cdf006,object_description,Give a concise description of this 3D object.,"<pad> A 3D model of a moose with a snout, a snout","<pad> A 3D model of a moose with a snout, a snout",ok,,1,1.000000,1.000000



Saved files:
- /content/3D-LLM/3DLLM_BLIP2-base/minimal_rigorous_results/feature_robustness_rows.csv
- /content/3D-LLM/3DLLM_BLIP2-base/minimal_rigorous_results/feature_robustness_summary.csv
- /content/3D-LLM/3DLLM_BLIP2-base/minimal_rigorous_results/prompt_robustness_rows.csv
- /content/3D-LLM/3DLLM_BLIP2-base/minimal_rigorous_results/prompt_robustness_summary.csv
